# Data scouting for comparator cities and NO₂ coverage

This notebook contains three exploratory checks that are not used directly to calculate the dissertation results. I keep them separately because they informed later choices about comparator cities and data availability.

The checks cover Lyon and Marseille station names, historical and recent Paris NO₂ coverage, and the feasibility of using London data.

## Data and references

**Data**
- [EEA Air Quality Download Service](https://www.eea.europa.eu/en/datahub/datahubitem-view/778ef9f5-6293-4846-badd-56a29c70880d): station metadata, Historical observations through 2012 and Verified (E1a) observations for 2013-2024.

**Data conventions**
- The annual coverage screen follows the [EEA indicator methodology for exceedances of air-quality standards](https://www.eea.europa.eu/en/analysis/indicators/exceedance-of-air-quality-standards). The EEA describes the criterion as at least 75% valid data per calendar year. The same paragraph then says more than 6,570 valid hours in a normal year or more than 6,588 in a leap year, although those counts equal exactly 75%. Because the headline criterion is explicit and the hourly wording is inconsistent at this one-hour boundary, I apply `>= 75%` to the unrounded annual capture.
- [EEA pollutant code 8 is NO2](https://dd.eionet.europa.eu/vocabulary/aq/pollutant/8). For this coverage check, validity codes 1, 2 and 3 count as observations because the [EEA validity vocabulary](https://dd.eionet.europa.eu/vocabulary/aq/observationvalidity/) labels all three as valid. Codes 2 and 3 identify values below the detection limit, but the concentration itself is not used in this presence check.

**Software**
- pandas: McKinney, W. (2010). Data structures for statistical computing in Python. Proceedings of the 9th Python in Science Conference, 56-61.
- [`airbase`](https://airbase.readthedocs.io/en/stable/) is an independent Python client for the EEA download service; its source code is available on [GitHub](https://github.com/JohnPaton/airbase).


## A. Check Lyon and Marseille station names

I first check how Lyon and Marseille are recorded in the EEA municipality field. The final comparator selection is carried out in notebook 04 using the full station-classification rule.

In [1]:
import calendar
import glob
import os

import airbase
import nest_asyncio
import pandas as pd

nest_asyncio.apply()

NO2_VALID_CODES = [1, 2, 3]
CAPTURE_THRESHOLD = 75

station_metadata = pd.read_csv("data/meta/stations.csv", low_memory=False)
fr_pm25 = station_metadata[
    (station_metadata["Country"] == "France")
    & (station_metadata["Air Pollutant"] == "PM2.5")
]
fr_no2 = station_metadata[
    (station_metadata["Country"] == "France")
    & (station_metadata["Air Pollutant"] == "NO2")
]
paris_no2_codes = sorted(
    fr_no2.loc[
        fr_no2["Municipality"].str.startswith("PARIS ", na=False),
        "Air Quality Station EoI Code",
    ].dropna().unique()
)
municipalities = fr_pm25["Municipality"].dropna().unique()

lyon_names = [name for name in municipalities if "LYON" in name.upper()]
mars_names = [name for name in municipalities if "MARSEILLE" in name.upper()]
print("Lyon-related municipalities:", lyon_names)
print("Marseille-related municipalities:", mars_names)

Lyon-related municipalities: ['LYON 7E ARRONDISSEMENT', 'LYON 3E ARRONDISSEMENT']
Marseille-related municipalities: ['MARSEILLE 8E ARRONDISSEMENT', 'MARSEILLE 15E ARRONDISSEMENT', 'MARSEILLE 4E ARRONDISSEMENT']


In [2]:
station_columns = [
    "Air Quality Station EoI Code",
    "Air Quality Station Name",
    "Air Quality Station Type",
    "Municipality",
    "Operational Activity Begin",
    "Operational Activity End",
]

for city in ["LYON", "MARSEILLE"]:
    city_mask = fr_pm25["Municipality"].str.upper().str.startswith(
        city, na=False
    )
    city_stations = fr_pm25[city_mask]
    print(f"\n=== {city} PM2.5 stations ===")
    print(
        city_stations[station_columns]
        .drop_duplicates()
        .sort_values("Operational Activity Begin")
        .to_string()
    )


=== LYON PM2.5 stations ===
       Air Quality Station EoI Code Air Quality Station Name Air Quality Station Type            Municipality Operational Activity Begin Operational Activity End
100316                      FR20062              LYON Centre               background  LYON 3E ARRONDISSEMENT        01/02/2007 00:00:00                      NaN
99862                       FR20017                  GERLAND               background  LYON 7E ARRONDISSEMENT        12/05/2020 00:00:00                      NaN

=== MARSEILLE PM2.5 stations ===
      Air Quality Station EoI Code Air Quality Station Name Air Quality Station Type                  Municipality Operational Activity Begin Operational Activity End
97863                      FR03006        MARSEILLE RABATAU                  traffic   MARSEILLE 8E ARRONDISSEMENT        16/12/1981 00:00:00                      NaN
98208                      FR03043      MARSEILLE 5 AVENUES               background   MARSEILLE 4E ARRONDISSEMENT   

## B. Check Paris NO₂ data availability

This section checks which Paris NO₂ stations appear in the EEA Historical and Verified collections. Paris is defined from the municipality field rather than from the `FR04` prefix, because that prefix also covers stations elsewhere in the wider Airparif network.

Annual capture is the number of unique valid hourly timestamps divided by the expected number of hours in that year, including 8,784 hours in leap years. Following the EEA's stated headline rule, a station-year passes when its unrounded capture is at least 75%. The table is rounded only for display. Before combining the two sources in a model, the units, timestamps and station identifiers at the 2012/2013 boundary would still need to be checked.

In [3]:
os.makedirs("data/raw_no2_hist", exist_ok=True)

client = airbase.AirbaseClient()
historical_request = client.request("Historical", "FR", poll="NO2")
historical_request.download(
    dir="data/raw_no2_hist", skip_existing=True
)

URLs    : 100%|██████████| 782/782 [00:00<00:00, 4.79kURL/s]
download: 0.00b [00:00, ?b/s]


In [4]:
historical_files = glob.glob(
    "data/raw_no2_hist/**/*.parquet", recursive=True
)
assert historical_files, "No Historical NO2 files were found"
print(len(historical_files), "files")

historical_sample = pd.read_parquet(historical_files[0])
print(historical_sample.columns.tolist())
print(historical_sample.head())

782 files
['Samplingpoint', 'Pollutant', 'Start', 'End', 'Value', 'Unit', 'AggType', 'Validity', 'Verification', 'ResultTime', 'DataCapture']
              Samplingpoint  Pollutant               Start  \
0  FR/SPO-FR02026_00008_100          8 2005-01-01 00:00:00   
1  FR/SPO-FR02026_00008_100          8 2005-01-01 01:00:00   
2  FR/SPO-FR02026_00008_100          8 2005-01-01 08:00:00   
3  FR/SPO-FR02026_00008_100          8 2005-01-01 09:00:00   
4  FR/SPO-FR02026_00008_100          8 2005-01-01 10:00:00   

                  End                  Value Unit AggType  Validity  \
0 2005-01-01 01:00:00   6.000000000000000000  NaN    hour         1   
1 2005-01-01 02:00:00   7.000000000000000000  NaN    hour         1   
2 2005-01-01 09:00:00  12.000000000000000000  NaN    hour         1   
3 2005-01-01 10:00:00  17.000000000000000000  NaN    hour         1   
4 2005-01-01 11:00:00  17.000000000000000000  NaN    hour         1   

   Verification ResultTime DataCapture  
0           NaN  

In [5]:
historical_no2 = pd.concat(
    (pd.read_parquet(file) for file in historical_files),
    ignore_index=True,
)
print("total rows:", len(historical_no2))

historical_no2["Pollutant"] = pd.to_numeric(
    historical_no2["Pollutant"], errors="coerce"
)
historical_no2 = historical_no2[
    (historical_no2["Pollutant"] == 8)
    & (historical_no2["AggType"] == "hour")
    & historical_no2["Validity"].isin(NO2_VALID_CODES)
].copy()
historical_no2["Start"] = pd.to_datetime(historical_no2["Start"])
historical_no2["year"] = historical_no2["Start"].dt.year

historical_no2["station"] = historical_no2["Samplingpoint"].str.extract(r"(FR\d+)")
paris_historical = historical_no2[
    historical_no2["station"].isin(paris_no2_codes)
].copy()
print("Paris municipality stations found:",
      sorted(paris_historical["station"].unique()))


def annual_capture(data):
    unique_hours = data.drop_duplicates(["station", "Start"])
    counts = unique_hours.groupby(["station", "year"]).size().unstack("year")
    expected_hours = pd.Series(
        {year: (8784 if calendar.isleap(int(year)) else 8760)
         for year in counts.columns}
    )
    return counts.div(expected_hours, axis="columns").mul(100)

historical_coverage = annual_capture(paris_historical)
print(historical_coverage.round(0))

total rows: 54536520
Paris municipality stations found: ['FR04004', 'FR04012', 'FR04014', 'FR04031', 'FR04037', 'FR04055', 'FR04060', 'FR04071', 'FR04118', 'FR04131', 'FR04141', 'FR04143', 'FR04160', 'FR04329']
year     1999  2000  2001  2002  2003  2004  2005  2006  2007  2008  2009  \
station                                                                     
FR04004  89.0  99.0  78.0  91.0  90.0  92.0  97.0  97.0  96.0  98.0  93.0   
FR04012  97.0  90.0  70.0  93.0  91.0  88.0  91.0  93.0  96.0  97.0  98.0   
FR04014  98.0  94.0  89.0  93.0  97.0  93.0  95.0  98.0  98.0  99.0  91.0   
FR04031  90.0  98.0  86.0  97.0  92.0  93.0  95.0  90.0  89.0  86.0  93.0   
FR04037  97.0  97.0  93.0  89.0  96.0  98.0  96.0  96.0  96.0  98.0  90.0   
FR04055   NaN   NaN  44.0  94.0  97.0  95.0  88.0  99.0  98.0  93.0  92.0   
FR04060  96.0  96.0  87.0  89.0  91.0  88.0  97.0  90.0  94.0  99.0  93.0   
FR04071  95.0  98.0  92.0  96.0  86.0  81.0  92.0  96.0  95.0  98.0  84.0   
FR04118   NaN   NaN

In [6]:
# FR04329 only has about 2% coverage in the final historical year.
print(historical_coverage.loc["FR04329"].round(1))

year
1999    NaN
2000    NaN
2001    NaN
2002    NaN
2003    NaN
2004    NaN
2005    NaN
2006    NaN
2007    NaN
2008    NaN
2009    NaN
2010    NaN
2011    NaN
2012    1.9
Name: FR04329, dtype: float64


In [7]:
os.makedirs("data/raw_no2_verified", exist_ok=True)
verified_request = client.request("Verified", "FR", poll="NO2")
verified_request.download(
    dir="data/raw_no2_verified", skip_existing=True
)

URLs    : 100%|██████████| 614/614 [00:00<00:00, 4.25kURL/s]
download: 0.00b [00:00, ?b/s]


In [8]:
# Apply the EEA at-least-75% rule to unrounded annual capture.
historical_window = historical_coverage.loc[
    :, [year for year in historical_coverage.columns if 1999 <= year <= 2012]
]
historical_eligible = historical_window[
    (historical_window >= CAPTURE_THRESHOLD).all(axis=1)
].index.tolist()
print(historical_eligible)

['FR04004', 'FR04014', 'FR04031', 'FR04037', 'FR04060', 'FR04071']


In [9]:
# Repeat the coverage check for the Verified period.
verified_files = glob.glob(
    "data/raw_no2_verified/**/*.parquet", recursive=True
)
assert verified_files, "No Verified NO2 files were found"
verified_no2 = pd.concat(
    (pd.read_parquet(file) for file in verified_files),
    ignore_index=True,
)
verified_no2["Pollutant"] = pd.to_numeric(
    verified_no2["Pollutant"], errors="coerce"
)
verified_no2 = verified_no2[
    (verified_no2["Pollutant"] == 8)
    & (verified_no2["AggType"] == "hour")
    & verified_no2["Validity"].isin(NO2_VALID_CODES)
].copy()
verified_no2["Start"] = pd.to_datetime(verified_no2["Start"])
verified_no2["year"] = verified_no2["Start"].dt.year
verified_no2["station"] = verified_no2["Samplingpoint"].str.extract(r"(FR\d+)")
paris_verified = verified_no2[
    verified_no2["station"].isin(paris_no2_codes)
].copy()

verified_coverage = annual_capture(paris_verified)
modern_window = verified_coverage.loc[
    :, [year for year in verified_coverage.columns if 2013 <= year <= 2024]
]
modern_eligible = modern_window[
    (modern_window >= CAPTURE_THRESHOLD).all(axis=1)
].index.tolist()
print("At least 75% in every year, 2013-2024:", modern_eligible)

bridge_stations = sorted(
    set(historical_eligible) & set(modern_eligible)
)
print("At least 75% in every year across both periods:", bridge_stations)

At least 75% in every year, 2013-2024: ['FR04004', 'FR04014', 'FR04031', 'FR04037', 'FR04060', 'FR04118', 'FR04131', 'FR04141', 'FR04329']
At least 75% in every year across both periods: ['FR04004', 'FR04014', 'FR04031', 'FR04037', 'FR04060']


### Interpretation of the NO₂ coverage check

A station appears in the bridge list only if at least 75% of its expected hourly observations are present in every year from 1999-2012 and again in every year from 2013-2024. This is a coverage screen rather than a claim of uninterrupted observations. It establishes whether a long-run Paris series may be feasible, but it does not combine the two EEA collections. Units, timestamps and station identifiers at the 2012/2013 boundary would need to be reconciled before modelling.

## C. Check whether the EEA data can support a London comparison

This final section checks whether the Great Britain files can identify London stations directly enough for a possible ULEZ comparison. The EEA metadata have blank municipality values for Great Britain and use generic sampling-point identifiers. Station names and coordinates are available, so a London crosswalk may be possible, but it is not built or validated here. The Verified EEA release also ends in 2024, one year before the project's 2025 endpoint.

In [10]:
os.makedirs("data/raw_london", exist_ok=True)
client = airbase.AirbaseClient()

for pollutant in ["PM2.5", "NO2"]:
    london_request = client.request(
        "Verified", "GB", poll=pollutant
    )
    london_request.download(
        dir="data/raw_london", skip_existing=True
    )

URLs    : 100%|██████████| 104/104 [00:00<00:00, 1.01kURL/s]
download: 0.00b [00:00, ?b/s]
URLs    : 100%|██████████| 174/174 [00:00<00:00, 1.62kURL/s]
download: 0.00b [00:00, ?b/s]


In [11]:
london_files = glob.glob(
    "data/raw_london/**/*.parquet", recursive=True
)
assert london_files, "No Great Britain files were found"
london_measurements = pd.concat(
    (pd.read_parquet(file) for file in london_files),
    ignore_index=True,
)
london_pollutants = set(
    pd.to_numeric(
        london_measurements["Pollutant"], errors="coerce"
    ).dropna().astype(int)
)
assert london_pollutants == {8, 6001}, (
    "The London folder contains an unexpected pollutant selection"
)
print("rows:", len(london_measurements))
print(london_measurements["Samplingpoint"].dropna().unique()[:20])

rows: 12802634
<ArrowStringArray>
['GB/GB_SamplingPoint_64830', 'GB/GB_SamplingPoint_66392',
 'GB/GB_SamplingPoint_65129', 'GB/GB_SamplingPoint_74622',
 'GB/GB_SamplingPoint_74632', 'GB/GB_SamplingPoint_63358',
 'GB/GB_SamplingPoint_76561', 'GB/GB_SamplingPoint_65337',
 'GB/GB_SamplingPoint_64727', 'GB/GB_SamplingPoint_74536',
 'GB/GB_SamplingPoint_74650',   'GB/GB_SamplingPoint_156',
 'GB/GB_SamplingPoint_65284', 'GB/GB_SamplingPoint_65318',
 'GB/GB_SamplingPoint_24460',   'GB/GB_SamplingPoint_428',
 'GB/GB_SamplingPoint_75951',    'GB/GB_SamplingPoint_78',
 'GB/GB_SamplingPoint_66533', 'GB/GB_SamplingPoint_61575']
Length: 20, dtype: str


In [12]:
client = airbase.AirbaseClient()
client.download_metadata("data/london_meta.csv")

Writing metadata to data/london_meta.csv...


In [13]:
london_metadata = pd.read_csv(
    "data/london_meta.csv", low_memory=False
)
print(london_metadata.columns.tolist())
print(london_metadata.head(3))

['Country', 'B-G Namespace', 'Year', 'Air Quality Network', 'Air Quality Network Name', 'Timezone', 'Air Quality Station EoI Code', 'Air Quality Station Nat Code', 'Air Quality Station Name', 'Sampling Point Id', 'Air Pollutant', 'Longitude', 'Latitude', 'Altitude', 'Altitude Unit', 'Air Quality Station Area', 'Air Quality Station Type', 'Operational Activity Begin', 'Operational Activity End', 'Sample Id', 'Inlet Height', 'Inlet Height Unit', 'Building Distance', 'Building Distance Unit', 'Kerb Distance', 'Kerb Distance Unit', 'Distance Source', 'Distance Source Unit', 'Main Emission Sources', 'Heating Emissions', 'Heating Emissions Unit', 'Mobile', 'Traffic Emissions', 'Traffic Emissions Unit', 'Industrial Emissions', 'Industrial Emissions Unit', 'Municipality', 'Dispersion Local', 'Dispersion Regional', 'Distance Junction', 'Distance Junction Unit', 'Heavy Duty Fraction', 'Height Facades', 'Street Width', 'Traffic Speed', 'Traffic Volume', 'Process Id', 'Process Activity Begin', 'Pr

In [14]:
gb_metadata = london_metadata[
    london_metadata["Country"] == "United Kingdom"
].copy()

print("GB rows:", len(gb_metadata))
print(gb_metadata["Air Quality Station Area"].value_counts().head())
print(gb_metadata["Municipality"].dropna().unique()[:40])
print("\nSampling Point Id format:",
      gb_metadata["Sampling Point Id"].dropna().unique()[:5])

location_fields = [
    "Air Quality Station Name", "Longitude", "Latitude"
]
print("\nShare of GB metadata rows with location fields:")
print(gb_metadata[location_fields].notna().mean().round(2))

GB rows: 11742
Air Quality Station Area
urban       7824
rural       2092
suburban    1826
Name: count, dtype: int64
<ArrowStringArray>
[]
Length: 0, dtype: str

Sampling Point Id format: <ArrowStringArray>
['GB_SamplingPoint_22558', 'GB_SamplingPoint_61843', 'GB_SamplingPoint_61844',
 'GB_SamplingPoint_61845', 'GB_SamplingPoint_65801']
Length: 5, dtype: str

Share of GB metadata rows with location fields:
Air Quality Station Name    1.0
Longitude                   1.0
Latitude                    1.0
dtype: float64


### London scouting conclusion

The Great Britain metadata do not provide municipality values and the sampling-point identifiers do not identify London directly. The station names and coordinates could be used to construct a crosswalk, but that would require a separate validation step and would still leave the missing 2025 EEA year. I therefore do not use London as a comparator in this dissertation. If this comparison were developed later, the [Defra UK-AIR archive](https://uk-air.defra.gov.uk/data/) or the [London Air Quality Network](https://www.londonair.org.uk/LondonAir/Default.aspx) would be more direct starting points.